# Person 2: Body, Clinical Measurements & DVC Exploration

**Author:** Rishabh (Person 2: Body, Clinical Measurements & DVC)  
**Course:** AI252TA — MLOps  
**Topic:** Clinical Measurements & Anthropometrics (NHANES 2017-2018 & 2021-2023)  

This notebook documents the clinical reasoning, exploratory analysis, and data cleaning decisions that power `src/data/body.py`.

## 1. Setup and Environment Verification

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.data.body import read_xpt, process_bmx, process_bp, process_bpq, process_labs, build, CONTRACT

raw_2017 = Path("data/raw/nhanes/2017-2018")
print("Environment initialized successfully.")

## 2. Inspecting Raw Blood Pressure & Auscultation Quirks
In NHANES 2017-2018, manual auscultation was used with up to 4 readings. Diastolic pressure has recordings of 0 mmHg (stored as 5.4e-79 in SAS), which represent unmeasured Korotkoff sounds rather than physiological zeros.

In [ ]:
bpx = pd.read_sas(raw_2017 / "BPX_J.xpt", format="xport")
print("Diastolic readings < 1e-5 (zeros):")
for i in range(1, 5):
    print(f"  BPXDI{i} zeros:", (bpx[f"BPXDI{i}"] < 1e-5).sum())

print("\nSummary of SBP readings 1 to 3:")
print(bpx[["BPXSY1", "BPXSY2", "BPXSY3"]].describe().round(1))

## 3. Anthropometric Consistency: Reported BMI vs Calculated BMI
Checking whether CDC reported BMXBMI matches weight_kg / (height_m)^2.

In [ ]:
bmx = pd.read_sas(raw_2017 / "BMX_J.xpt", format="xport")
calc_bmi = bmx["BMXWT"] / ((bmx["BMXHT"] / 100) ** 2)
diff = (calc_bmi - bmx["BMXBMI"]).abs()
print("Maximum difference:", round(diff.max(), 4), "kg/m^2")
print("Mean difference:", round(diff.mean(), 4), "kg/m^2 (due solely to CDC rounding to 1 decimal place)")

## 4. Run Topic Transformation Pipeline for 2017-2018
Executing `build('2017-2018')` and verifying output against the team data contract.

In [ ]:
body_df = build("2017-2018")
print("Output shape:", body_df.shape)
print("Columns match contract:", list(body_df.columns) == CONTRACT)
body_df.head()

## 5. Critical Leakage Finding: HbA1c vs Diabetes
HbA1c >= 6.5% is the clinical gold standard for diagnosing diabetes. Since diabetes is part of Person 1's prediction target, HbA1c represents direct target leakage and must not be used as an input feature in predictive modeling.

In [ ]:
people_df = pd.read_csv("data/interim/people_2017-2018.csv")
merged = people_df.merge(body_df, on="seqn", how="inner")

plt.figure(figsize=(8, 5))
sns.boxplot(data=merged.dropna(subset=["hba1c", "diabetes"]), x="diabetes", y="hba1c", palette=["#3b82f6", "#ef4444"], hue="diabetes", legend=False)
plt.axhline(6.5, color="red", linestyle="--", linewidth=2, label="ADA Diagnostic Cutoff (6.5%)")
plt.title("HbA1c Target Leakage Demonstration (Adults 20+)")
plt.xlabel("Doctor Diagnosed Diabetes (0 = No, 1 = Yes)")
plt.ylabel("HbA1c (%)")
plt.legend()
plt.tight_layout()
plt.show()